In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import sys
from logging import INFO, WARNING, ERROR, StreamHandler, getLogger

logger = getLogger()
if not logger.hasHandlers():
    logger.addHandler(StreamHandler(sys.stdout))
logger.setLevel(INFO)

# Import

In [ ]:
import copy
import math
import os
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm
import matplotlib.gridspec as gridspec

from src.simulation.seasonal_sro import SeasonalSROConfig, simulate_SeasonalSRO

from src.simulation.langevin2d_helper import (
    get_langevin_without_forcing_for_seasonal_SRO,
    InformationThermodynamicsResult,
    calc_diff_bound,
)
from src.analysis.peak_detector import detect_peak_counts_scipy
from src.utils.io_pickles import read_pickle

plt.rcParams["font.family"] = "serif"
plt.style.use("tableau-colorblind10")

# Define constants

In [ ]:
ROOT_DIR = str((pathlib.Path(os.environ["PYTHONPATH"].split(":")[0]) / "..").resolve())
DATA_KIND = "KA21"  # "KA21", "V25", "H26"
assert DATA_KIND in ["KA21", "V25", "H26"]

In [ ]:
DEFAULT = SeasonalSROConfig(value_set_name=DATA_KIND)

In [ ]:
df = pd.DataFrame.from_dict(DEFAULT.__dict__, orient="index", columns=["value"])
df.loc["WBJ (Re. Part)", "value"] = DEFAULT.get_bwj_real_part()
df.loc["WBJ (Im. Part)", "value"] = DEFAULT.get_bwj_imaginary_part()
df.loc["Intrinsic Period"] = 2 * math.pi / DEFAULT.get_bwj_imaginary_part()
display(df)
print(np.round(df, 4).to_markdown())

In [ ]:
FIG_DIR = "./fig"
os.makedirs(FIG_DIR, exist_ok=True)

# Simulate Seasonal RO

## Growth Rate

In [ ]:
t_min = 0
t_max = 1  # unit: year
n_timesteps = int(t_max) * 360

dt = (t_max - t_min) / n_timesteps
ts = torch.linspace(t_min, t_max, n_timesteps + 1, dtype=torch.float64)

config = copy.deepcopy(DEFAULT)
# config.phi = 0.0
Rt = config.get_Rt(ts)

plt.rcParams["font.size"] = 18
fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(1, 1, 1)

ax.plot(ts * 360, Rt, color="#ABABAB", lw=3)
ax.set_xlabel("Calendar Month")
ax.set_ylabel(r"Growth Rate, $R(t)$ [1/yr]")

ax.set_xticks(np.linspace(0, 360, 13)[:12] + 15, labels=np.arange(1, 13))
ax.set_xlim(0, 360)
for i, t in enumerate(np.linspace(0, 360, 13)):
    if t + 15 > 360:
        break
    ax.axvline(t + 15, color="gray", linestyle="--", alpha=0.3)
ax.axhline(0, color="k", linestyle="-", alpha=1.0)

if (
    DATA_KIND == "KA21WeakNoise"
    or DATA_KIND == "KA21"
    or DATA_KIND == "KA21StrongNoise"
):
    ax.set_ylim(-3, 1)
if DATA_KIND == "V25":
    ax.set_ylim(-4, 2)
if DATA_KIND == "H26":
    ax.set_ylim(-2, 3)

fig.tight_layout()
plt.show()

if DATA_KIND == "KA21":
    fig.savefig(f"{FIG_DIR}/fig01.jpg", bbox_inches="tight")
    fig.savefig(f"{FIG_DIR}/fig01.pdf", bbox_inches="tight")

del config, t_min, t_max, n_timesteps, dt

## Std and Ensemble Members

In [ ]:
n_batches = 10_000
t_min = 0.0
t_max = 100  # unit: year
n_timesteps = int((t_max - t_min) * 360)
dict_results = {}

for is_seasonality_on in [True, False]:

    config = copy.deepcopy(DEFAULT)
    if not is_seasonality_on:
        config.Ra = 0.0
        seed = 314
    else:
        seed = 42

    # Spin Up
    x0 = torch.tensor([0.0, 10.0], dtype=torch.float64).unsqueeze(0)  # shape (1, 2)
    x0 = x0.repeat(n_batches, 1)  # shape (n_batches, 2)
    # x0 = torch.randn_like(x0) # use this if you want random initial conditions
    _, xs = simulate_SeasonalSRO(
        x0=x0,
        t_min=t_min,
        t_max=t_max,
        n_timesteps=n_timesteps,
        n_batches=n_batches,
        config=config,
        seed=seed,
    )
    x0 = xs[:, -1].clone().detach()
    del xs

    ts, xs = simulate_SeasonalSRO(
        x0=x0,
        t_min=t_min,
        t_max=t_max,
        n_timesteps=n_timesteps,
        n_batches=n_batches,
        config=config,
        seed=seed + 1000,
    )

    dict_results[is_seasonality_on] = {
        "ts": ts.clone().detach(),
        "T": xs[:, :, 0].clone().detach(),
        "h": xs[:, :, 1].clone().detach(),
        "std_T": xs[:, :, 0].std(dim=0),
        "std_h": xs[:, :, 1].std(dim=0),
        "cov": (xs[:, :, 0] * xs[:, :, 1]).mean(dim=0),
    }
    del xs, ts, config

In [ ]:
n = 360 * 5
i_members = {}
if DATA_KIND == "H26":
    i_members["seasonal_on"] = 24  # 3, 5, 24, 33
    i_members["seasonal_off"] = 40  # 25, 33, 39, 40
elif DATA_KIND == "KA21":
    i_members["seasonal_on"] = 24  # 3, 23, 24, 26
    i_members["seasonal_off"] = 39  # 4, 6, 25, 39, 55
else:
    i = 15
    i_members["seasonal_on"] = i
    i_members["seasonal_off"] = i

plt.rcParams["font.size"] = 19
fig, axes = plt.subplots(5, 2, figsize=(14, 15))

for j, is_seasonality_on in enumerate([False, True]):
    T = dict_results[is_seasonality_on]["T"]
    h = dict_results[is_seasonality_on]["h"]
    std_T = dict_results[is_seasonality_on]["std_T"]
    std_h = dict_results[is_seasonality_on]["std_h"]
    cov = dict_results[is_seasonality_on]["cov"]
    ts = dict_results[is_seasonality_on]["ts"]

    if is_seasonality_on:
        i_member = i_members["seasonal_on"]
    else:
        i_member = i_members["seasonal_off"]

    if j == 0:
        sub_ttl = "(a) Steady SRO, $R_a = 0$"
    else:
        sub_ttl = "(b) Unsteady SRO, $R_a = 2.0 |R_0|$"
        if DATA_KIND == "H26":
            sub_ttl = "(b) Unsteady SRO, $R_a = 5.3 |R_0|$"

    for i, (ax, label, color, data) in enumerate(
        zip(
            axes[:, j],
            [
                "$T$ [K]",
                r"std. $T$ [K]",
                "$h$ [m]",
                r"std. $h$ [m]",
                r"cov [K m]",
            ],
            ["#006BA4", "#006BA4", "#FF800E", "#FF800E", "black"],
            [T, std_T, h, std_h, cov],
        )
    ):
        xs = ts.numpy()[-n:] - ts.numpy()[-n]
        ls = "-" if "std" in label else "--"

        if "std" in label or "cov" in label:
            ys = data.numpy()[-n:]
            ax.plot(xs, ys, color=color, linestyle=ls, lw=4)
        else:
            ys = data.numpy()[i_member, -n:]
            ax.plot(xs, ys, color=color, linestyle=ls, lw=2)

        if j == 0:
            ax.set_ylabel(label, fontsize=19)
        if j == 1:
            ax.set_yticklabels([])

        if i == 4:
            ax.set_xlabel("Time [yr]", fontsize=18)

        if "$T$" in label and "std" in label:
            if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
                ax.set_yticks(np.linspace(0.5, 1.1, 5))
                ax.set_ylim(0.5, 1.1)
            if DATA_KIND == "KA21StrongNoise":
                ax.set_yticks(np.linspace(1.6, 4.0, 5))
                ax.set_ylim(1.6, 4.0)
            if DATA_KIND == "V25":
                ax.set_yticks(np.linspace(1.0, 2.6, 5))
                ax.set_ylim(1.0, 2.6)
            if DATA_KIND == "H26":
                ax.set_yticks(np.linspace(0.4, 1.1, 5))
                ax.set_ylim(0.4, 1.1)
            ax.set_title(r"Standard Deviation (std.): $T$")
        elif "$h$" in label and "std" in label:
            if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
                ax.set_yticks(np.linspace(4.9, 5.7, 5))
                ax.set_ylim(4.9, 5.7)
            if DATA_KIND == "KA21StrongNoise":
                ax.set_yticks(np.linspace(17, 20, 5))
                ax.set_ylim(17, 20)
            if DATA_KIND == "V25":
                ax.set_yticks(np.linspace(14, 18, 5))
                ax.set_ylim(14, 18)
            if DATA_KIND == "H26":
                ax.set_yticks(np.linspace(5.3, 6.9, 5))
                ax.set_ylim(5.3, 6.9)
            ax.set_title(r"Standard Deviation (std.): $h$")
        elif "$T$" in label:
            if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
                ax.set_yticks(np.linspace(-3.0, 3.0, 5))
                ax.set_ylim(-3, 3)
            if DATA_KIND == "KA21StrongNoise":
                ax.set_yticks(np.linspace(-10.0, 10.0, 5))
                ax.set_ylim(-10, 10)
            if DATA_KIND == "V25":
                ax.set_yticks(np.linspace(-5, 5.0, 5))
                ax.set_ylim(-5, 5)
            if DATA_KIND == "H26":
                ax.set_yticks(np.linspace(-3, 3.0, 5))
                ax.set_ylim(-3, 3)
            ax.axhline(0, color="k", linestyle="-", alpha=1.0)
            ax.set_title(sub_ttl + "\n\n" + r"One Realization: $T$")
        elif "$h$" in label:
            if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
                ax.set_yticks(np.linspace(-20, 20, 5))
                ax.set_ylim(-20, 20)
            if DATA_KIND == "KA21StrongNoise":
                ax.set_yticks(np.linspace(-60, 60, 5))
                ax.set_ylim(-60, 60)
            if DATA_KIND == "V25":
                ax.set_yticks(np.linspace(-50, 50, 5))
                ax.set_ylim(-50, 50)
            if DATA_KIND == "H26":
                ax.set_yticks(np.linspace(-20, 20, 5))
                ax.set_ylim(-20, 20)
            ax.axhline(0, color="k", linestyle="-", alpha=1.0)
            ax.set_title(r"One Realization: $h$")
        elif "cov" in label:
            if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
                ax.set_yticks([-0.5, 0.0, 1.0, 2.0])
                ax.set_ylim(-0.5, 2.0)
                pass
            if DATA_KIND == "KA21StrongNoise":
                ax.set_yticks([-5, 0, 10, 20])
                ax.set_ylim(-5, 20)
            if DATA_KIND == "V25":
                ax.set_yticks(np.linspace(-10, 10, 3))
                ax.set_ylim(-10, 10)
            if DATA_KIND == "H26":
                ax.set_yticks(np.linspace(-5, 0, 3))
                ax.set_ylim(-5, 0)
            ax.axhline(0, color="k", linestyle="-", alpha=1.0)
            ax.set_title(r"Covariance: $\left\langle T h \right\rangle$")

        for yr in np.linspace(0, 5, 11):
            ls = "-" if yr in [0, 1, 2, 3, 4, 5] else "--"
            ax.axvline(yr, color="gray", linestyle=ls, alpha=0.3)
        ax.set_xlim(0, 5)

plt.tight_layout()
plt.show()

if DATA_KIND == "KA21":
    fig.savefig(f"{FIG_DIR}/fig02.jpg", bbox_inches="tight")
    fig.savefig(f"{FIG_DIR}/fig02.pdf", bbox_inches="tight")
elif DATA_KIND == "H26":
    fig.savefig(f"{FIG_DIR}/fig07.jpg", bbox_inches="tight")
    fig.savefig(f"{FIG_DIR}/fig07.pdf", bbox_inches="tight")
del T, h, std_T, std_h, ts, xs

# TUR

In [ ]:
initial_month = 3  # or 9, results are almost independent of initial_month
assert 1 <= initial_month <= 12

t_min = float(initial_month - 1) / 12.0
t_max = 200  # unit: year
nt_year = 360  # number of days in a year
n_timesteps = int((t_max - t_min) * nt_year)

dict_results = {}
lst_amp_Ra = [0.5, 1.0, 2.0]
if DATA_KIND == "H26":
    lst_amp_Ra = [1.0, 3.0, 6.0]

for amp_Ra in lst_amp_Ra:
    logger.info(f"Calculating for {amp_Ra=}")

    config = copy.deepcopy(DEFAULT)
    config.Ra = abs(config.R0) * amp_Ra

    mu0 = torch.tensor([0.0, 10.0], dtype=torch.float64)
    # results are almost independent of mu0
    # mu0 = torch.randn_like(mu0)  # use this for random initialization

    ts, langevin = get_langevin_without_forcing_for_seasonal_SRO(
        cfg=config, t_min=t_min, t_max=t_max, n_timesteps=n_timesteps, mu0=mu0
    )
    result = InformationThermodynamicsResult(langevin)

    mean_abs_T = torch.mean(langevin.x_ave()[-nt_year - 1 : -1].abs()).item()
    assert mean_abs_T < 1e-7  # mean is almost zero

    dict_results[amp_Ra] = {
        "month": np.linspace(0, 12, nt_year, endpoint=False),
        "var_T": (langevin.x2_ave() - langevin.x_ave() ** 2)[-nt_year - 1 : -1],
        "Rt": config.get_Rt(ts)[-nt_year - 1 : -1],
        "tur_bound_for_dvar_dt_sq": torch.sqrt(result.bound_dvar_dt_sq[-nt_year:]),
        "dvar_dt_sq": torch.sqrt(result.dvar_dt_sq[-nt_year:]),  # dx_dt_cdot_x
        "sum_of_all": (result.sX + result.qX - result.iflX)[-nt_year:].clone().detach(),
        "Q": result.qX[-nt_year:].clone().detach(),
        "S": result.sX[-nt_year:].clone().detach(),
        "-I": -result.iflX[-nt_year:].clone().detach(),
        "P": result.tur_pt[-nt_year:].clone().detach(),
        "current": result.tur_current[-nt_year:].clone().detach(),
        "langevin": langevin,
        "all_results": result,
        "ts": ts,
    }
    del config, mu0, ts, langevin, result, mean_abs_T

In [ ]:
plt.rcParams["font.size"] = 19
fig, all_axes = plt.subplots(4, 3, sharex=False, figsize=(17, 15))

lst_amp_Ra = [0.5, 1.0, 2.0]
if DATA_KIND == "H26":
    lst_amp_Ra = [1.0, 3.0, 6.0]

for j, amp_Ra in enumerate(lst_amp_Ra):
    axes = all_axes[:, j]
    d = dict_results[amp_Ra]

    ax = axes[0]
    ax.plot(d["month"], d["Rt"], color="#ABABAB", lw=2)
    if j == 0:
        ax.set_ylabel(r"$R(t)$ [1/yr]")
    ax.axhline(0.0, ls="-", color="k", lw=1.0)
    if (
        DATA_KIND == "KA21WeakNoise"
        or DATA_KIND == "KA21"
        or DATA_KIND == "KA21StrongNoise"
    ):
        ax.set_ylim(-3, 1)
    if DATA_KIND == "V25":
        ax.set_ylim(-4, 2)
    if DATA_KIND == "H26":
        ax.set_ylim(-2, 3)

    if j == 0:
        if DATA_KIND == "H26":
            assert amp_Ra == 1.0
            ax.set_title(
                r"(a) $R_a = 1.0 |R_0|$"
                + "\n\n"
                + "Growth Rate, $R(t)$"
            )
        else:
            assert amp_Ra == 0.5
            ax.set_title(
                r"(a) $R_a = 0.5 |R_0|$"
                + "\n\n"
                + "Growth Rate, $R(t)$"
            )
    elif j == 1:
        if DATA_KIND == "H26":
            assert amp_Ra == 3.0
            ax.set_title(
                r"(b) $R_a = 3.0 |R_0|$"
                + "\n\n"
                + "Growth Rate, $R(t)$"
            )
        else:
            assert amp_Ra == 1.0
            ax.set_title(
                r"(b) $R_a = 1.0 |R_0|$"
                + "\n\n"
                + "Growth Rate, $R(t)$"
            )
    elif j == 2:
        if DATA_KIND == "H26":
            assert amp_Ra == 6.0
            ax.set_title(
                r"(c) $R_a = 6.0 |R_0|$"
                + "\n\n"
                + "Growth Rate, $R(t)$"
            )
        else:
            assert amp_Ra == 2.0
            ax.set_title(
                r"(c) $R_a = 2.0 |R_0|$"
                + "\n\n"
                + "Growth Rate, $R(t)$"
            )

    color_T = "#006BA4"
    ax = axes[1]
    ax.plot(d["month"], d["var_T"], color=color_T, lw=3)
    if j == 0:
        ax.set_ylabel(r"var. $T$ [K$^2$]")
    if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
        ax.set_ylim(0.1, 1.1)
        ax.set_yticks(np.linspace(0.1, 1.1, 6))
    if DATA_KIND == "KA21StrongNoise":
        ax.set_ylim(2, 14)
        ax.set_yticks(np.linspace(2, 14, 7))
    if DATA_KIND == "V25":
        ax.set_ylim(0, 7)
        ax.set_yticks(np.linspace(0, 7, 8))
    if DATA_KIND == "H26":
        ax.set_ylim(0.2, 1.2)
        ax.set_yticks(np.linspace(0.2, 1.2, 6))
    ax.set_title(r"Variance of $T$ ($= \langle T^2\rangle$)")

    ax = axes[2]
    l = r"$\dot{\mathcal{P}}$ ($=\frac{2\dot{\mathcal{Q}}}{(\sigma^T)^2}+\frac{{\rm d}\mathcal{S}}{{\rm d}t}-\dot{\mathcal{I}}$)"
    ax.plot(d["month"], d["P"], color="#C85200", ls="--", label=l, lw=2)
    print(f'{torch.min(d["P"])=}')
    l = r"$\left(\frac{1}{2}(\sigma^T)^2\langle T^2\rangle \right)^{-1} \left(\frac{1}{2}\frac{{\rm d}}{{\rm d}t} \langle T^2 \rangle\right)^2$"
    ax.plot(d["month"], d["current"], color="k", label=l, lw=3)
    print(f'{torch.min(d["current"])=}')
    if j == 0:
        ax.set_ylabel(r"Change Rate [1/yr]")
    if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
        ax.set_ylim(-5.0, 20.0)
        ax.set_yticks(np.linspace(-5, 20, 6))
    if DATA_KIND == "KA21StrongNoise":
        ax.set_ylim(-5.0, 20.0)
        ax.set_yticks(np.linspace(-5, 20, 6))
    if DATA_KIND == "V25":
        ax.set_ylim(-5.0, 20.0)
        ax.set_yticks(np.linspace(-5, 20, 6))
    if DATA_KIND == "H26":
        ax.set_ylim(-5.0, 30.0)
        ax.set_yticks(np.linspace(-5, 30, 8))
    ax.set_title("TUR")
    ax.axhline(0.0, ls="-", color="k", lw=1.0)
    if j == 0:
        ax.legend(ncol=1, fontsize=18)

    ax = axes[3]
    # l = r"$\dot{\mathcal{P}}_{T}$ ($=\frac{{\rm d}\mathcal{S}_T}{{\rm d}t}+\frac{2\dot{\mathcal{Q}}_T}{\sigma_T^2}-\dot{\mathcal{I}}_T$)"
    # ax.plot(d["month"], d["sum_of_all"], color="r", ls="--", label=l, lw=2)
    ax.plot(
        d["month"],
        d["Q"],
        label=r"$\frac{2\dot{\mathcal{Q}}}{(\sigma^T)^2}$",
        ls=":",
        lw=3,
        color="#FFBC79",
    )
    ax.plot(
        d["month"],
        d["S"],
        label=r"$\frac{{\rm d}\mathcal{S}}{{\rm d}t}$",
        ls="--",
        lw=3,
        color="#A2C8EC",
    )
    ax.plot(d["month"], d["-I"], label=r"$-\dot{\mathcal{I}}$", lw=3, color="#C85200")
    ax.axhline(0.0, ls="-", color="k", lw=0.5)
    if j == 0:
        ax.set_ylabel("Change Rate [1/yr]")
    if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
        ax.set_ylim(-5.0, 20.0)
        ax.set_yticks(np.linspace(-5, 20, 6))
    if DATA_KIND == "KA21StrongNoise":
        ax.set_ylim(-5.0, 20.0)
        ax.set_yticks(np.linspace(-5, 20, 6))
    if DATA_KIND == "V25":
        ax.set_ylim(-5.0, 20.0)
        ax.set_yticks(np.linspace(-5, 20, 6))
    if DATA_KIND == "H26":
        ax.set_ylim(-5.0, 30.0)
        ax.set_yticks(np.linspace(-5, 30, 8))
    ax.set_xlabel("Calendar Month")
    ax.set_title("TUR Terms")
    if j == 0:
        ax.legend(ncol=3, fontsize=18)
    if j != 0:
        for ax in axes.flatten():
            ax.set_yticklabels([])

    for ax in axes.flatten():
        ax.set_xlim(0, 12)
        ax.set_xticks(
            np.linspace(1, 12, 12) - 0.5, np.linspace(1, 12, 12, dtype=np.int32)
        )
        # ax.set_xticklabels([])
    # axes.flatten()[-1].set_xticks(
    #     np.linspace(1, 12, 12) - 0.5, np.linspace(1, 12, 12, dtype=np.int32)
    # )

plt.tight_layout()
plt.show()

if DATA_KIND == "KA21":
    fig.savefig(f"{FIG_DIR}/fig03.jpg", bbox_inches="tight")
    fig.savefig(f"{FIG_DIR}/fig03.pdf", bbox_inches="tight")

In [ ]:
nt_year = 360  # number of days in a year
plt.rcParams["font.size"] = 19
fig, all_axes = plt.subplots(4, 3, sharey=True, sharex=False, figsize=(17, 15))

lst_amp_Ra = [0.5, 1.0, 2.0]
if DATA_KIND == "H26":
    lst_amp_Ra = [1.0, 3.0, 6.0]

for j, amp_Ra in enumerate(lst_amp_Ra):
    axes = all_axes[:, j]
    d = dict_results[amp_Ra]
    langevin = d["langevin"]

    ax = axes[0]
    if j == 0:
        if DATA_KIND == "H26":
            assert amp_Ra == 1.0
            ax.set_title(
                r"(a) $R_a = 1.0 |R_0|$"
                + "\n\n"
                + r"$A=(2/(\sigma^T)^2) R(t)^2 \left\langle T^2 \right\rangle$"
            )
        else:
            assert amp_Ra == 0.5
            ax.set_title(
                r"(a) $R_a = 0.5 |R_0|$"
                + "\n\n"
                + r"$A=(2/(\sigma^T)^2) R(t)^2 \left\langle T^2 \right\rangle$"
            )
    elif j == 1:
        if DATA_KIND == "H26":
            assert amp_Ra == 3.0
            ax.set_title(
                r"(b) $R_a = 3.0 |R_0|$"
                + "\n\n"
                + r"$A=(2/(\sigma^T)^2) R(t)^2 \left\langle T^2 \right\rangle$"
            )
        else:
            assert amp_Ra == 1.0
            ax.set_title(
                r"(b) $R_a = 1.0 |R_0|$"
                + "\n\n"
                + r"$A=(2/(\sigma^T)^2) R(t)^2 \left\langle T^2 \right\rangle$"
            )
    elif j == 2:
        if DATA_KIND == "H26":
            assert amp_Ra == 6.0
            ax.set_title(
                r"(c) $R_a = 6.0 |R_0|$"
                + "\n\n"
                + r"$A=(2/(\sigma^T)^2) R(t)^2 \left\langle T^2 \right\rangle$"
            )
        else:
            assert amp_Ra == 2.0
            ax.set_title(
                r"(c) $R_a = 2.0 |R_0|$"
                + "\n\n"
                + r"$A=(2/(\sigma^T)^2) R(t)^2 \left\langle T^2 \right\rangle$"
            )

    q = (langevin.Qx_over_Tx())[-nt_year:]

    a = (langevin.a11**2) * langevin.x2_ave() / langevin.Tx
    a = a[-nt_year - 1 : -1]

    b = 2 * langevin.xy_ave() * (langevin.a11 * langevin.a12) / langevin.Tx
    b = b[-nt_year - 1 : -1]

    c = (langevin.a12**2) * langevin.y2_ave() / langevin.Tx
    c = c[-nt_year - 1 : -1]

    e = langevin.a11
    e = e[-nt_year - 1 : -1]

    s = a + b + c + e

    for i, (l, data) in enumerate(zip(["A", "B", "C", "D"], [a, b, c, e])):

        ax = axes[i]
        label = r"$\frac{2\dot{\mathcal{Q}}}{(\sigma^T)^2}=A+B+C+D$"
        ax.plot(d["month"], q, label=label, ls=":", lw=3, color="#FFBC79")
        ax.plot(d["month"], data, color="k", lw=2, label=l)
        # ax.plot(d["month"], s, color="gray")

        if j == 0:
            ax.legend(ncol=2, fontsize=18, loc="upper right")
        if i == 1:
            ax.set_title(
                r"$B = (2/(\sigma^T)^2) 2 R(t) F_1 \left\langle Th \right\rangle$"
            )
        elif i == 2:
            ax.set_title(
                r"$C = (2/(\sigma^T)^2) (F_1)^2 \left\langle h^2 \right\rangle$"
            )
        elif i == 3:
            ax.set_title(r"$D = R(t)$")

        if j == 0:
            ax.set_ylabel("Change Rate [1/yr]")

        if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
            ax.set_ylim(-5.0, 20.0)
            ax.set_yticks(np.linspace(-5, 20, 6))
        if DATA_KIND == "KA21StrongNoise":
            ax.set_ylim(-5.0, 20.0)
            ax.set_yticks(np.linspace(-5, 20, 6))
        if DATA_KIND == "V25":
            ax.set_ylim(-5.0, 20.0)
            ax.set_yticks(np.linspace(-5, 20, 6))
        if DATA_KIND == "H26":
            ax.set_ylim(-10.0, 40.0)
            ax.set_yticks([-10, 0, 10, 20, 30, 40])

    ax = axes[-1]
    ax.set_xlabel("Calendar Month")
    for ax in axes.flatten():
        ax.axhline(0.0, ls="-", color="k", lw=0.5)
        ax.set_xlim(0, 12)
        ax.set_xticks(
            np.linspace(1, 12, 12) - 0.5, np.linspace(1, 12, 12, dtype=np.int32)
        )
plt.tight_layout()
plt.show()

if DATA_KIND == "KA21":
    fig.savefig(f"{FIG_DIR}/fig05.jpg", bbox_inches="tight")
    fig.savefig(f"{FIG_DIR}/fig05.pdf", bbox_inches="tight")
elif DATA_KIND == "H26":
    fig.savefig(f"{FIG_DIR}/fig08.jpg", bbox_inches="tight")
    fig.savefig(f"{FIG_DIR}/fig08.pdf", bbox_inches="tight")

# Dependence on Ra

In [ ]:
peak_mode = "all_peaks"  # "max_only", "all_peaks"
t_max = 10_000  # year

is_random_init = False
n_months_for_running_mean = 5  # 3 or 5
threshold_n_months = 6  # 7 or 6

dk = DATA_KIND
out_dir = f"{ROOT_DIR}/data/models/SRO_long_run_{t_max:06}yr_{dk}"
if is_random_init:
    out_dir += "_random_init"

lst_amp_Ra = np.linspace(0, 2.5, 26) if dk != "H26" else np.linspace(0, 6.0, 61)
n_per_month = 30  # days in a month
nt_year = 12 * n_per_month  # days in a year
peak_counts = np.zeros(shape=(len(lst_amp_Ra), nt_year), dtype=np.int32)
lst_bounds = []

for idx, amp in tqdm(enumerate(lst_amp_Ra), total=len(lst_amp_Ra)):
    out_file_path = f"{out_dir}/SRO_Ra{str(amp).replace('.', 'p')[:3]}.pickle"
    data = read_pickle(out_file_path)

    years = data["ts"]
    time_series = pd.Series(data["xs"][0, :, 0])  # SST time series
    assert len(years) == len(time_series)

    smoothed = time_series.rolling(
        window=n_months_for_running_mean * n_per_month + 1, center=True, win_type=None
    ).mean()

    logger.setLevel(WARNING)

    elnino_counts = detect_peak_counts_scipy(
        years=years,
        smoothed_series=smoothed,
        std_value=float(np.std(time_series)),
        n_per_month=n_per_month,
        mode="elnino",
        peak_mode=peak_mode,
        peak_min_distance=int(n_per_month * n_months_for_running_mean),
        peak_min_width=int(n_per_month * n_months_for_running_mean),
        threshold_n_months=threshold_n_months,
    )
    assert len(elnino_counts) == nt_year

    peak_counts[idx] = elnino_counts

    config = copy.deepcopy(DEFAULT)
    config.Ra = abs(config.R0) * amp
    diff_bound, _, min_bound = calc_diff_bound(
        t_min=0.0, t_max=200.0, nt_year=nt_year, config=config
    )
    lst_bounds.append(diff_bound)

    logger.setLevel(INFO)
    del data, years, time_series, smoothed
    del elnino_counts, diff_bound, min_bound, config

In [ ]:
width = 10  # number of days in each bin

results, centers = [], []
days = np.arange(0, 360 + width, width)
assert days[0] == 0 and days[-1] == 360

for start in days:
    end = start + width
    if end > 360:
        break
    data = peak_counts[:, start:end]
    assert data.shape == (len(lst_amp_Ra), width), f"{data.shape=}, {start=}, {end=}"
    results.append(np.sum(data, axis=1))
    centers.append((start + end) // 2 + 1)

peaks = np.stack(results, axis=1)
probs = peaks / np.sum(peaks, axis=1, keepdims=True)
diff_probs = np.max(probs, axis=1) - np.min(probs, axis=1)
assert np.all(np.abs(np.sum(probs, axis=1) - 1.0) < 1e-7)

xs = centers
ys = lst_amp_Ra
X, Y = np.meshgrid(xs, ys, indexing="ij")

plt.rcParams["font.size"] = 14
fig = plt.figure(figsize=(13, 6))
gs = gridspec.GridSpec(1, 5, figure=fig)

ax = fig.add_subplot(gs[0, 1:3])

ret = ax.pcolormesh(
    X, Y, probs.transpose(), shading="nearest", cmap="magma", vmin=0.0, vmax=0.12
)
cbar = fig.colorbar(ret, ax=ax, extend="max")
ax.set_title("Relative Frequency")

# ax.set_ylabel(f"Amplitude of $R_a$ ($= |R_a/R_0|$)")
ax.set_ylim(min(lst_amp_Ra) - 0.05, max(lst_amp_Ra) + 0.05)
ax.set_xticks(
    np.linspace(0, 360, 13)[:12] + 15,
    np.arange(1, 13),
)
ax.set_xlabel("Calendar Month")
# ax.set_yticks([])

if DATA_KIND == "H26":
    ax.axhline(5.33, color="w", ls="--")
else:
    ax.axhline(2.0, color="w", ls="--")

ax = fig.add_subplot(gs[0, 0:1])
ax.plot(lst_bounds, lst_amp_Ra, color="k")
print(min(lst_bounds))
ax.set_ylim(min(lst_amp_Ra) - 0.05, max(lst_amp_Ra) + 0.05)
ax.set_xlabel(
    r"$\Delta \dot{\cal P}$ ($= \max \;\dot{\cal P} - \min \;\dot{\cal P}$) [1/yr]"
)
ax.set_ylabel(f"Magnitude of $R_a$ ($= |R_a/R_0|$)")
if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
    ax.set_xticks([0, 1.2, 2.4])
    ax.set_xlim(-0.1, 2.4)
elif DATA_KIND == "H26":
    ax.set_xticks([0, 1, 2, 3])
    ax.set_xlim(-0.1, 3.0)
elif DATA_KIND == "V25":
    ax.set_xticks([0, 1, 2, 3])
    ax.set_xlim(-0.1, 3)

ax = fig.add_subplot(gs[0, 3:4])
ax.plot(np.sum(peaks, axis=1) / t_max, lst_amp_Ra, color="k")
ax.set_ylim(min(lst_amp_Ra) - 0.05, max(lst_amp_Ra) + 0.05)
ax.set_xlabel(r"Num. of Peaks [1/yr]")
# ax.set_yticks([])
if DATA_KIND == "KA21WeakNoise" or DATA_KIND == "KA21":
    ax.set_xticks([0.12, 0.125, 0.130])
    ax.set_xlim(0.12, 0.13)
if DATA_KIND == "KA21StrongNoise":
    ax.set_xticks([0.12, 0.125, 0.130])
    ax.set_xlim(0.12, 0.13)
if DATA_KIND == "V25":
    ax.set_xticks([0.11, 0.12, 0.130])
    ax.set_xlim(0.11, 0.13)
if DATA_KIND == "H26":
    ax.set_xticks([0.12, 0.125, 0.130])
    ax.set_xlim(0.12, 0.13)
plt.tight_layout()
plt.show()

if DATA_KIND == "KA21":
    fig.savefig(f"{FIG_DIR}/fig04.jpg", bbox_inches="tight")
    fig.savefig(f"{FIG_DIR}/fig04.pdf", bbox_inches="tight")